In [1]:
import time
notebook_start = time.perf_counter()

import os, json, pandas as pd, numpy as np, joblib
from thermoift import print_model_metrics
from thermoift.rng_utils import get_rng
from sklearn.model_selection import train_test_split, cross_validate

In [2]:
PLOT_FOLDER = "TabPFN_gamma_OUTPUTS"
target      = "gamma"
SEED        = 50005
TEST_ROWS   = None
DATA_PATH   = ""


In [3]:
# Parameters
PLOT_FOLDER = "/scratch-shared/draju/PART_2/ACTIVELEARNING/OUTPUTS/AL_ST/N025/trial_02/Gamma"
TEST_ROWS = None
SEED = 52225
DATA_PATH = "/scratch-shared/draju/PART_2/ACTIVELEARNING/COMBINED/AL_ST/N025/trial_02.csv"


In [4]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="TabPFN fit/predict failed at leaf",
    category=UserWarning,
    module="tabpfn",
)

try:
    import tabpfn
    from tabpfn import TabPFNRegressor
    from tabpfn.constants import ModelVersion

    os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

    print(f"TabPFN version: {tabpfn.__version__}")
    print("Selected model version:", ModelVersion.V2)

except ImportError as exc:
    raise ImportError("tabpfn is not installed in this Python environment.") from exc

n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))
os.environ["OMP_NUM_THREADS"]      = str(n_cpus)
os.environ["MKL_NUM_THREADS"]      = str(n_cpus)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cpus)
os.environ["NUMEXPR_NUM_THREADS"]  = str(n_cpus)
try:
    import torch
    torch.set_num_threads(n_cpus)
except ImportError:
    pass
print(f"Thread limit set to {n_cpus} (SLURM_CPUS_PER_TASK)")

TabPFN version: 7.1.1
Selected model version: ModelVersion.V2
Thread limit set to 4 (SLURM_CPUS_PER_TASK)


In [5]:
df = pd.read_csv(DATA_PATH)
print(f"Number of rows: {len(df)}")

if TEST_ROWS is not None:
    df = df.iloc[:TEST_ROWS].copy()
    print(f"Test mode: using first {TEST_ROWS} rows only")
else:
    print("Full mode: using all rows")

print(f"Total samples: {len(df)}")
print(f"\n{target} statistics:")
print(df[target].describe())

Number of rows: 4863
Full mode: using all rows
Total samples: 4863

gamma statistics:
count    4863.000000
mean        9.594959
std         6.086363
min         0.153596
25%         4.144664
50%         9.211144
75%        14.556542
max        22.539230
Name: gamma, dtype: float64


In [6]:
rng       = get_rng(seed=SEED)

z_columns  = [col for col in df.columns if col.startswith("z_")]
Z_non_zero = [col for col in z_columns if (df[col] != 0).any()]
features   = ["temperature", "pressure"] + Z_non_zero

print(f"Selected features: {features}")

X = df[features]
y = df[target]

# 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_test,  X_val,  y_test,  y_val  = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Testing samples:    {X_test.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")

Selected features: ['temperature', 'pressure', 'z_carbon dioxide', 'z_hydrogen', 'z_nitrogen', 'z_argon', 'z_methane', 'z_oxygen', 'z_carbon monoxide', 'z_hydrogen sulfide']

Training samples:   3404
Testing samples:    729
Validation samples: 730


In [7]:
# Train TabPFN model
tabpfn_model = TabPFNRegressor(random_state=SEED, ignore_pretraining_limits=True, fit_mode="fit_preprocessors")
tabpfn_model.fit(X_train, y_train)

y_train_pred = tabpfn_model.predict(X_train)
y_test_pred  = tabpfn_model.predict(X_test)
y_val_pred   = tabpfn_model.predict(X_val)

metrics = print_model_metrics(y_train, y_train_pred, y_test, y_test_pred, target, unit="mN/m", y_val=y_val, y_val_pred=y_val_pred)

Model Performance for gamma

Training Set:
  R²:   0.999966
  RMSE: 0.035542 mN/m
  MAE:  0.025888 mN/m

Test Set:
  R²:   0.999957
  RMSE: 0.039558 mN/m
  MAE:  0.028826 mN/m

Validation Set:
  R²:   0.999954
  RMSE: 0.041533 mN/m
  MAE:  0.030485 mN/m


In [8]:
results_df = pd.DataFrame({
    "idx":       np.concatenate([y_train.index, y_test.index, y_val.index]),
    "actual":    np.concatenate([y_train.values, y_test.values, y_val.values]),
    "predicted": np.concatenate([y_train_pred,  y_test_pred,  y_val_pred]),
    "split":     ["train"]*len(y_train) + ["test"]*len(y_test) + ["val"]*len(y_val),
})
os.makedirs(PLOT_FOLDER, exist_ok=True)
results_df.to_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_predictions.csv"), index=False)
print(f"Predictions saved: {len(results_df)} rows")

model_path = os.path.join(PLOT_FOLDER, f"TabPFN_{target}_model.joblib")
joblib.dump(tabpfn_model, model_path)
print(f"Model saved to: {model_path}")

Predictions saved: 4863 rows
Model saved to: /scratch-shared/draju/PART_2/ACTIVELEARNING/OUTPUTS/AL_ST/N025/trial_02/Gamma/TabPFN_gamma_model.joblib


In [9]:
cv_results = cross_validate(
    tabpfn_model, X, y, cv=5,
    scoring={
        "r2":   "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae":  "neg_mean_absolute_error",
    },
    n_jobs=1,
)

cv_r2_scores   = cv_results["test_r2"]
cv_rmse_scores = -cv_results["test_rmse"]
cv_mae_scores  = -cv_results["test_mae"]

print(f"Cross-Validation R² Scores:   {cv_r2_scores}")
print(f"Mean CV R²:   {cv_r2_scores.mean():.6f} (+/- {cv_r2_scores.std() * 2:.6f})")
print(f"\nCross-Validation RMSE Scores: {cv_rmse_scores}")
print(f"Mean CV RMSE: {cv_rmse_scores.mean():.6f} (+/- {cv_rmse_scores.std() * 2:.6f})")
print(f"\nCross-Validation MAE Scores:  {cv_mae_scores}")
print(f"Mean CV MAE:  {cv_mae_scores.mean():.6f} (+/- {cv_mae_scores.std() * 2:.6f})")

Cross-Validation R² Scores:   [0.99774488 0.99669834 0.99491876 0.993528   0.99447055]
Mean CV R²:   0.995472 (+/- 0.003067)

Cross-Validation RMSE Scores: [0.29266209 0.34504376 0.43032134 0.50025106 0.44378483]
Mean CV RMSE: 0.402413 (+/- 0.148052)

Cross-Validation MAE Scores:  [0.21681513 0.25839143 0.26426234 0.38502511 0.28804129]
Mean CV MAE:  0.282507 (+/- 0.112323)


In [10]:
metrics["cv_r2_scores"]   = cv_r2_scores.tolist()
metrics["cv_r2_mean"]     = float(cv_r2_scores.mean())
metrics["cv_r2_std"]      = float(cv_r2_scores.std())
metrics["cv_rmse_scores"] = cv_rmse_scores.tolist()
metrics["cv_rmse_mean"]   = float(cv_rmse_scores.mean())
metrics["cv_rmse_std"]    = float(cv_rmse_scores.std())
metrics["cv_mae_scores"]  = cv_mae_scores.tolist()
metrics["cv_mae_mean"]    = float(cv_mae_scores.mean())
metrics["cv_mae_std"]     = float(cv_mae_scores.std())
metrics["model"]          = "TabPFN"
metrics["features"]       = features
metrics["target"]         = target
metrics["seed"]           = SEED

os.makedirs(PLOT_FOLDER, exist_ok=True)
metrics_path = os.path.join(PLOT_FOLDER, f"TabPFN_{target}_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"\nMetrics saved to: {metrics_path}")


Metrics saved to: /scratch-shared/draju/PART_2/ACTIVELEARNING/OUTPUTS/AL_ST/N025/trial_02/Gamma/TabPFN_gamma_metrics.json


In [11]:
notebook_end = time.perf_counter()
elapsed_minutes = (notebook_end - notebook_start) / 60
print(f"Total notebook runtime: {elapsed_minutes:.2f} minutes")

Total notebook runtime: 0.48 minutes
